# Drug-Drug Interaction — Exploratory Data Analysis
This notebook explores the BioSNAP-TWOSIDES dataset and visualizes the drug interaction graph.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import sys
sys.path.append('..')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Imports done')

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/raw/TWOSIDES.csv')
df.columns = ['drug1', 'drug2', 'side_effect_id', 'side_effect_name']
print(f'Dataset shape: {df.shape}')
print(f'\nColumn types:')
print(df.dtypes)
print(f'\nSample rows:')
df.head(10)

## 2. Basic Statistics

In [ ]:
n_drugs = pd.concat([df['drug1'], df['drug2']]).nunique()
n_pairs = df.groupby(['drug1','drug2']).ngroups
n_se    = df['side_effect_name'].nunique()
possible = n_drugs * (n_drugs - 1) / 2

print(f'Total rows:           {len(df):>10,}')
print(f'Unique drugs:         {n_drugs:>10,}')
print(f'Unique drug pairs:    {n_pairs:>10,}')
print(f'Unique side effects:  {n_se:>10,}')
print(f'Possible drug pairs:  {possible:>10,.0f}')
print(f'Graph density:        {n_pairs/possible*100:>9.1f}%')
print(f'Unknown pairs:        {possible-n_pairs:>10,.0f} ({(1-n_pairs/possible)*100:.1f}%)')
print(f'\nThe model predicts interactions for the unknown {(1-n_pairs/possible)*100:.1f}% of pairs')

## 3. Top Side Effects

In [ ]:
top_se = df['side_effect_name'].value_counts().head(20)

plt.figure(figsize=(12, 7))
sns.barplot(x=top_se.values, y=top_se.index, palette='viridis')
plt.title('Top 20 Most Common Polypharmacy Side Effects', fontsize=14, fontweight='bold')
plt.xlabel('Number of Drug Pairs')
plt.ylabel('')
plt.tight_layout()
plt.savefig('../data/processed/top_side_effects.png', dpi=150)
plt.show()
print(f'Most common: {top_se.index[0]} ({top_se.values[0]:,} drug pairs)')

## 4. Drug Interaction Degree Distribution
How many interactions does each drug have?

In [ ]:
pairs = df[['drug1', 'drug2']].drop_duplicates()
degree = pd.concat([pairs['drug1'], pairs['drug2']]).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(degree.values, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(degree.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {degree.mean():.0f}')
axes[0].axvline(degree.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {degree.median():.0f}')
axes[0].set_title('Drug Degree Distribution')
axes[0].set_xlabel('Number of interactions per drug')
axes[0].set_ylabel('Count')
axes[0].legend()

top_drugs = degree.head(20)
axes[1].barh(top_drugs.index[::-1], top_drugs.values[::-1], color='steelblue')
axes[1].set_title('Top 20 Most Connected Drugs')
axes[1].set_xlabel('Number of interactions')

plt.suptitle('Drug Connectivity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/degree_distribution.png', dpi=150)
plt.show()

print(f'Min degree:    {degree.min()}')
print(f'Max degree:    {degree.max()}')
print(f'Mean degree:   {degree.mean():.1f}')
print(f'Median degree: {degree.median():.1f}')

## 5. Side Effects per Drug Pair

In [ ]:
se_per_pair = df.groupby(['drug1', 'drug2'])['side_effect_name'].count()

plt.figure(figsize=(10, 5))
plt.hist(se_per_pair.values, bins=50, color='coral', edgecolor='white')
plt.axvline(se_per_pair.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {se_per_pair.mean():.0f}')
plt.title('Number of Side Effects per Drug Pair', fontsize=13, fontweight='bold')
plt.xlabel('Side effects per pair')
plt.ylabel('Number of drug pairs')
plt.legend()
plt.tight_layout()
plt.savefig('../data/processed/se_per_pair.png', dpi=150)
plt.show()

print(f'Min side effects per pair: {se_per_pair.min()}')
print(f'Max side effects per pair: {se_per_pair.max()}')
print(f'Mean:                      {se_per_pair.mean():.1f}')

## 6. Morgan Fingerprint Visualization
Each drug is represented as a 2048-bit binary vector capturing its molecular substructure.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

cache = pd.read_csv('../data/processed/smiles_cache.csv', index_col=0)
valid = cache['smiles'].dropna()
print(f'Valid SMILES: {len(valid)}/{len(cache)}')

sample_smiles = valid.iloc[0]
mol = Chem.MolFromSmiles(sample_smiles)
fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
fp_array = np.array(fp)

plt.figure(figsize=(14, 3))
plt.bar(range(len(fp_array)), fp_array, color='steelblue', width=1.0)
plt.title(f'Morgan Fingerprint (2048 bits) — Sample Drug\nSMILES: {sample_smiles[:70]}...', fontsize=11)
plt.xlabel('Bit position')
plt.ylabel('Active (1) / Inactive (0)')
plt.tight_layout()
plt.savefig('../data/processed/fingerprint_viz.png', dpi=150)
plt.show()

print(f'Active bits: {fp_array.sum()} / 2048 ({fp_array.mean()*100:.1f}%)')
print(f'Most fingerprints are sparse — only ~5% of bits active per drug')

## 7. Fingerprint Similarity Between Drugs

In [ ]:
from rdkit.Chem import DataStructs

# Compute pairwise Tanimoto similarity for a sample of 50 drugs
sample_smiles_list = valid.iloc[:50]
fps = []
for smi in sample_smiles_list:
    mol = Chem.MolFromSmiles(str(smi))
    if mol:
        fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048))

# Compute similarity matrix
n = len(fps)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, cmap='Blues', xticklabels=False, yticklabels=False)
plt.title('Tanimoto Similarity Matrix — Sample of 50 Drugs', fontsize=13, fontweight='bold')
plt.xlabel('Drug index')
plt.ylabel('Drug index')
plt.tight_layout()
plt.savefig('../data/processed/similarity_matrix.png', dpi=150)
plt.show()

off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f'Mean pairwise similarity: {off_diag.mean():.3f}')
print(f'Max pairwise similarity:  {off_diag.max():.3f}')
print(f'Most drug pairs are chemically dissimilar — GNN must use graph structure to capture functional relationships')

## 8. Graph Object Summary

In [ ]:
data = torch.load('../data/processed/ddi_graph.pt', weights_only=False)

print('PyG Data Object Summary')
print('-'*40)
print(f'Nodes (drugs):         {data.num_nodes}')
print(f'Edges (interactions):  {data.num_edges}')
print(f'Node feature dim:      {data.x.shape[1]}')
print(f'Node feature matrix:   {data.x.shape}')
print(f'Edge index shape:      {data.edge_index.shape}')
print(f'\nFeature matrix memory: {data.x.element_size() * data.x.nelement() / 1024 / 1024:.1f} MB')
print(f'Avg degree:            {data.num_edges / data.num_nodes:.1f} interactions per drug')

## 9. Model Results Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['MLP Baseline', 'GCN', 'GAT'],
    'ROC-AUC': [0.9584, 0.9334, 0.9222],
    'Avg Precision': [0.9602, 0.9398, 0.9266],
    'Uses Graph': ['No', 'Yes', 'Yes']
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2ecc71', '#3498db', '#9b59b6']

bars1 = axes[0].bar(results['Model'], results['ROC-AUC'], color=colors)
axes[0].set_ylim(0.88, 0.97)
axes[0].set_title('ROC-AUC by Model', fontsize=12, fontweight='bold')
axes[0].set_ylabel('ROC-AUC')
for bar, val in zip(bars1, results['ROC-AUC']):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:.4f}', ha='center', fontweight='bold')

bars2 = axes[1].bar(results['Model'], results['Avg Precision'], color=colors)
axes[1].set_ylim(0.88, 0.97)
axes[1].set_title('Average Precision by Model', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Precision')
for bar, val in zip(bars2, results['Avg Precision']):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:.4f}', ha='center', fontweight='bold')

plt.suptitle('Model Comparison — Drug-Drug Interaction Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150)
plt.show()

print('Key finding: MLP outperforms GNNs')
print('Reason: Morgan fingerprints are highly expressive; graph aggregation causes over-smoothing on this dense graph')
print('This is consistent with known GNN limitations on dense homogeneous graphs')